# Model Input Inspection

Inspect persisted training-preparation artifacts through the shared MotifML contracts.
The notebook can read a real Parquet-backed `05_model_input` root when `MOTIFML_MODEL_INPUT_ROOT`
points at one, or fall back to the tracked representative-row snapshots under
`tests/fixtures/training/representative_rows/`.


In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from motifml.datasets.model_input_storage import coerce_model_input_storage_schema
from motifml.datasets.tokenized_model_input_runtime_dataset import (
    TokenizedModelInputRuntimeDataset,
)
from motifml.training.contracts import (
    ModelInputMetadata,
    coerce_split_manifest_entries,
)
from motifml.training.data_loading import (
    LoadedTokenizedDocument,
    SpecialTokenIds,
    build_token_window_example,
)
from motifml.training.model_input import TokenizedDocumentRow
from motifml.training.special_token_policy import coerce_special_token_policy
from motifml.training.token_codec import (
    coerce_frozen_vocabulary,
    decode_token_ids_to_strings,
)

pd.set_option("display.max_colwidth", 120)

repo_root = Path.cwd()
fixture_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_FIXTURE_ROOT",
        repo_root / "tests" / "fixtures" / "training",
    )
).resolve()
model_input_root_env = os.environ.get("MOTIFML_MODEL_INPUT_ROOT")
model_input_root = (
    None if model_input_root_env is None else Path(model_input_root_env).resolve()
)
requested_document = os.environ.get(
    "MOTIFML_MODEL_INPUT_DOCUMENT",
    "ensemble_polyphony_controls.json",
)


def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


def load_documents() -> (
    tuple[list[LoadedTokenizedDocument], ModelInputMetadata, object, str]
):
    if model_input_root is not None and (model_input_root / "records").exists():
        runtime_handle = TokenizedModelInputRuntimeDataset(str(model_input_root)).load()
        documents: list[LoadedTokenizedDocument] = []
        for split_name in ("train", "validation", "test"):
            documents.extend(
                list(
                    runtime_handle.build_document_dataset(
                        split=split_name,
                        iteration_options={
                            "shuffle_shards": False,
                            "shuffle_documents": False,
                            "shuffle_windows": False,
                        },
                    )
                )
            )
        return (
            documents,
            runtime_handle.metadata,
            runtime_handle.storage_schema,
            "Runtime Dataset",
        )

    metadata = ModelInputMetadata.from_json_dict(
        load_json(fixture_root / "model_input" / "parameters.json")
    )
    storage_schema = coerce_model_input_storage_schema(
        load_json(fixture_root / "model_input" / "storage_schema.json")
    )
    documents = [
        LoadedTokenizedDocument(
            shard_id="global",
            record_path=path.relative_to(fixture_root).as_posix(),
            row=TokenizedDocumentRow.from_row_dict(load_json(path)),
        )
        for path in sorted((fixture_root / "representative_rows").rglob("*.row.json"))
        if path.is_file()
    ]
    return documents, metadata, storage_schema, "Representative Row Snapshots"


split_manifest = coerce_split_manifest_entries(
    load_json(fixture_root / "split_manifest.json")
)
split_stats = load_json(fixture_root / "split_stats.json")
vocabulary_payload = load_json(fixture_root / "vocabulary.json")
vocabulary = coerce_frozen_vocabulary(vocabulary_payload)
model_input_stats = load_json(fixture_root / "model_input_stats.json")
documents, metadata, storage_schema, document_source = load_documents()

display(
    Markdown(
        "## Model-Input Overview\n\n"
        f"- Source: `{document_source}`\n"
        f"- Documents: `{len(documents)}`\n"
        f"- Vocabulary Size: `{len(vocabulary.id_to_token)}`\n"
        f"- Model Input Version: `{metadata.model_input_version}`\n"
        f"- Storage Schema: `{storage_schema.storage_schema_version}` (`{storage_schema.backend}`)\n"
        f"- Requested Document: `{requested_document}`\n"
    )
)

In [ ]:
split_counts = Counter(entry.split.value for entry in split_manifest)
display(
    Markdown(
        "## Split Manifest\n\n"
        + "\n".join(
            f"- {split_name}: `{split_counts.get(split_name, 0)}` documents"
            for split_name in ("train", "validation", "test")
        )
        + f"\n- Split Version: `{split_stats['split_version']}`\n"
    )
)
display(
    pd.DataFrame(
        [
            {
                "relative_path": entry.relative_path,
                "document_id": entry.document_id,
                "split": entry.split.value,
                "group_key": entry.group_key,
            }
            for entry in split_manifest
        ]
    )
)

In [ ]:
top_tokens = vocabulary_payload["token_counts"][:10]
display(
    Markdown(
        "## Vocabulary Summary\n\n"
        f"- Vocabulary Version: `{vocabulary_payload['vocabulary_version']}`\n"
        f"- Feature Version: `{vocabulary_payload['feature_version']}`\n"
        f"- Token Count Used For Freeze: `{vocabulary_payload['token_count']}`\n"
        f"- Special Tokens: `{', '.join(vocabulary.id_to_token[:4])}`\n"
    )
)
display(pd.DataFrame(top_tokens))

In [ ]:
distribution_rows = [
    {
        "relative_path": document.row.relative_path,
        "split": document.row.split.value,
        "shard_id": document.shard_id,
        "token_count": document.row.token_count,
        "window_count": len(document.row.window_start_offsets),
        "window_offsets": ", ".join(
            str(offset) for offset in document.row.window_start_offsets
        ),
    }
    for document in documents
]
distribution_frame = pd.DataFrame(distribution_rows).sort_values(
    by=["split", "relative_path"],
    kind="stable",
)
shard_balance = (
    distribution_frame.groupby("shard_id", dropna=False)[
        ["token_count", "window_count"]
    ]
    .sum()
    .reset_index()
)
display(
    Markdown(
        "## Tokenized Document Distribution\n\n"
        f"- Splits Present: `{', '.join(sorted(distribution_frame['split'].unique()))}`\n"
        f"- Shards: `{', '.join(sorted(distribution_frame['shard_id'].unique()))}`\n"
        f"- Max Token Count: `{distribution_frame['token_count'].max()}`\n"
        f"- Max Window Count: `{distribution_frame['window_count'].max()}`\n"
    )
)
display(distribution_frame)
display(shard_balance)

In [ ]:
document_by_path = {document.row.relative_path: document for document in documents}
selected_document = document_by_path.get(requested_document, documents[0])
special_token_ids = SpecialTokenIds.from_vocabulary(vocabulary)
special_token_policy = coerce_special_token_policy(
    selected_document.row.special_token_policy
)

window_examples = [
    build_token_window_example(
        selected_document.row,
        shard_id=selected_document.shard_id,
        window_index=index,
        window_start_offset=offset,
        special_token_ids=special_token_ids,
    )
    for index, offset in enumerate(selected_document.row.window_start_offsets)
]
window_preview_rows = []
for example in window_examples[:3]:
    input_tokens = decode_token_ids_to_strings(example.input_ids, vocabulary=vocabulary)
    target_tokens = decode_token_ids_to_strings(
        example.target_ids, vocabulary=vocabulary
    )
    validated_tokens = special_token_policy.validate_window_tokens(
        input_tokens,
        padding_strategy=selected_document.row.padding_strategy,
        is_first_window=example.window_index == 0,
        is_last_window=example.window_index == len(window_examples) - 1,
    )
    window_preview_rows.append(
        {
            "window_index": example.window_index,
            "window_start_offset": example.window_start_offset,
            "attention_tokens": int(sum(example.attention_mask)),
            "input_preview": " ".join(input_tokens[:8]),
            "target_preview": " ".join(target_tokens[:8]),
            "validated_semantic_preview": " ".join(validated_tokens[:8]),
        }
    )

display(
    Markdown(
        "## Window Reconstruction\n\n"
        f"- Document: `{selected_document.row.relative_path}`\n"
        f"- Split / Shard: `{selected_document.row.split.value}` / `{selected_document.shard_id}`\n"
        f"- Token Count: `{selected_document.row.token_count}`\n"
        f"- Window Offsets: `{', '.join(str(offset) for offset in selected_document.row.window_start_offsets) or 'none'}`\n"
        f"- Padding Strategy: `{selected_document.row.padding_strategy}`\n"
        f"- BOS/EOS Policy: `{special_token_policy.bos_placement.value}` / `{special_token_policy.eos_placement.value}`\n"
    )
)
display(pd.DataFrame(window_preview_rows))